In [10]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
import numpy as np
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeFez
backend = FakeFez()
shots = 8192

#Credits to: https://qiskit.github.io/qiskit-aer/tutorials/2_device_noise_simulation.html

In [ ]:
def circuit(basis):    
# 1. Register definition
    q = QuantumRegister(1, 'q')
    cbit = ClassicalRegister(1, 'cbit')
    circ = QuantumCircuit(q, cbit)

    # 2. Initialization (Probabilities 3/4 and 1/4)
    theta = np.pi / 3
    circ.ry(theta, 0)

    # 8. Final DATA measurement
    if basis == 'X':
        circ.h(0)
    elif basis == 'Y':
        circ.sdg(0)
        circ.h(0)
    circ.measure([0], cbit)     


    # 9. Execution on the simulator
    sim_ideal = AerSimulator()
    result = sim_ideal.run(transpile(circ, sim_ideal), shots=8192).result()
    counts = result.get_counts(circ)

    if basis == 'X':
        fig1 = plot_histogram(counts, title='Ideal X-basis Measurement')
    elif basis == 'Y':
        fig2 = plot_histogram(counts, title='Ideal Y-basis Measurement')
    else:
        fig3 = plot_histogram(counts, title='Ideal Z-basis Measurement')
    return counts

In [9]:
basis = ['Z', 'X', 'Y']
measures = []
for b in basis:
    measures.append(circuit(b))



counts_0_z_ideal = sum(val for key, val in measures[0].items() if key.startswith('0'))
counts_1_z_ideal = sum(val for key, val in measures[0].items() if key.startswith('1'))
mean_z_ideal = (counts_0_z_ideal - counts_1_z_ideal) / shots


counts_0_x_ideal = sum(val for key, val in measures[1].items() if key.startswith('0'))
counts_1_x_ideal = sum(val for key, val in measures[1].items() if key.startswith('1'))
mean_x_ideal = (counts_0_x_ideal - counts_1_x_ideal) / shots

counts_0_y_ideal = sum(val for key, val in measures[2].items() if key.startswith('0'))
counts_1_y_ideal = sum(val for key, val in measures[2].items() if key.startswith('1'))
mean_y_ideal = (counts_0_y_ideal - counts_1_y_ideal) / shots


# Print the results
print(f"Mean value for Z component: {mean_z_ideal}")
print(f"Mean value for X component: {mean_x_ideal}")
print(f"Mean value for Y component: {mean_y_ideal}")

# Verify probability conservation
prob = mean_x_ideal ** 2 + mean_y_ideal ** 2 + mean_z_ideal ** 2
print(f"Probability conservation (should be close to 1.0): {prob}")

Mean value for Z component: 0.504150390625
Mean value for X component: 0.868896484375
Mean value for Y component: 0.01025390625
Probability conservation (should be close to 1.0): 1.0092538595199585
